<a href="https://colab.research.google.com/github/RohanYashraj/ifoa-workshop/blob/main/notebooks_v3/04_genai_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 · GenAI Tools Demo

* **Workshop:** AI for Actuaries
* **Session / Part:** S1.P2
* **Slides covered:** S1.P2.23, S1.P2.24, S1.P2.25
* **Author:** Satya Sai Mudigonda, Dr Rohan Yashraj Gupta (FIA, FIAI) and Sai Krishna Vadali
* **Workshop date:** 25 September 2026 · Gurgaon

## What this notebook does

Three reproducible Gemini API prompts that mirror Demos 1, 2, and 3 from the slide deck.

Demo 4 (Cursor IDE) is a live screen-share, not a notebook cell.

## Prerequisites

- Google account (for Colab)
- Gemini API key — set in **Colab Secrets** as `GEMINI_API_KEY` (free tier is sufficient)
- No local install required

## How to run

Top menu → **Runtime → Run all**. The first cell installs `google-genai`; the rest run without intervention.


## 0. Install

In [1]:
# Install google-genai. Pinned at 1.0+ — confirm against your Colab runtime
# at notebook freeze time.
!pip install -q google-genai

## 0.1 Standard imports and reproducibility

In [2]:
# === Standard imports ===
import json
import os
import datetime as dt
from pathlib import Path
from IPython.display import Markdown, display

# Reproducibility (LLM outputs are still non-deterministic by default —
# we set temperature=0 in calls below to get closer-to-stable results).
SEED = 42

# Notebook-specific imports below
# -------------------------------------------------------------
from google import genai
from google.genai import types

## 0.2 API setup

We read the Gemini API key from **Colab Secrets**. Add it in the left sidebar (key icon)
under the name `GOOGLE_API_KEY`. Local fallback: set the environment variable.

Pinning the model identifier matters — `gemini-2.5-flash` today is not the same model
in six months. Always pin in code.


In [3]:
# Read GOOGLE_API_KEY from Colab Secrets, fall back to env var.
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    if "GOOGLE_API_KEY" not in os.environ:
        raise RuntimeError(
            "GOOGLE_API_KEY is not set. "
            "Add it via Colab Secrets (left sidebar key icon) "
            "or set the env variable."
        )

# Pin the model. Confirm at notebook-finalisation time and update if the
# stable identifier has changed.
# MODEL_ID = "gemini-2.5-flash"
MODEL_ID = "gemini-3.1-flash-lite"

# Single shared client.
client = genai.Client()

print(f"API key loaded. Model pinned to: {MODEL_ID}")


API key loaded. Model pinned to: gemini-3.1-flash-lite


## 1. Demo 1 — risk-factor table for ABC Motor (GI)

**Slide:** S1.P2.23
**Goal:** Generate ten rating factors for an ABC Motor private car comprehensive
product, each tagged with directional impact and a one-line justification.
**Output format:** JSON

> ABC Insurer is hypothetical — see `00_personas_and_datasets.md`. Numbers are
> illustrative.


In [4]:
import json
from google import genai

# Create Gemini client
client = genai.Client()

# Select model
MODEL_ID = "gemini-3.1-flash-lite"

# Prompt
prompt = """
Generate 10 rating factors for an Indian private car insurance product.

For each factor return:
- factor
- direction ("increase" or "decrease")
- justification

Return ONLY valid JSON.
"""

# Generate structured output
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config={
        "response_mime_type": "application/json",
    },
)

# Parse JSON
factors = json.loads(response.text)

# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

# Print formatted output
print(json.dumps(factors, indent=2))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE
[
  {
    "factor": "Age of the Vehicle",
    "direction": "decrease",
    "justification": "As a vehicle ages, its Insured Declared Value (IDV) decreases, leading to a lower premium for own damage coverage."
  },
  {
    "factor": "Geographic Zone",
    "direction": "increase",
    "justification": "Vehicles registered in Tier-1 cities (Zone A) face higher premiums due to increased traffic density and higher accident/theft risk compared to smaller towns."
  },
  {
    "factor": "Engine Capacity (CC)",
    "direction": "increase",
    "justification": "Vehicles with higher engine cubic capacity have higher repair costs and are statistically associated with higher accident severity."
  },
  {
    "factor": "No Claim Bonus (NCB)",
    "direction": "decrease",
    "justification": "A cumulative discount is applied for every claim-free year, significantly reducing the renewal premium for safe drivers."
  },
  {
    "factor": "Car Fuel Type",
    "direction": "increa

## 2. Demo 2 — UW guidelines for ABC Term Life (Life)

**Slide:** S1.P2.24
**Goal:** Draft underwriting guidelines for an ABC Term Life product covering
issue ages 25–55 and sums assured up to ₹1 crore (Indian retail market).
**Output format:** Markdown, 300–400 words

> Output is a draft. Numerical thresholds (NoMed/TeleMed/FullMed bands etc.) must
> be matched against ABC Life's live underwriting matrix before publication.


In [5]:
from google import genai

# Create Gemini client
client = genai.Client()

# Select model
MODEL_ID = "gemini-3.1-flash-lite"

# Prompt
prompt = """
Draft underwriting guidelines for an Indian term life insurance product.

Include:
1. Eligibility
2. Medical underwriting
3. Financial underwriting
4. Smoker rules
5. PED handling

Assume:
- Entry age: 25–55
- Sum assured: up to ₹1 crore

Use simple markdown format.
"""

# Generate response
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
)

# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

# Render markdown nicely in Colab
display(Markdown(response.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE


These underwriting guidelines are designed for a standard individual term life insurance product in the Indian market, covering a Sum Assured (SA) of up to ₹1 crore.

---

### 1. Eligibility Criteria
*   **Entry Age:** 25 to 55 years (last birthday).
*   **Maximum Maturity Age:** 75 years.
*   **Residency:** Must be an Indian Resident (citizenship/valid Aadhaar/PAN required).
*   **Employment:** Salaried or Self-Employed with verifiable income proof.
*   **Sum Assured Limits:** 
    *   Minimum: ₹25 Lakhs.
    *   Maximum: ₹1 Crore.

---

### 2. Medical Underwriting
The medical requirements are triggered based on age and the Sum Assured (SA).

*   **Standard Cases:** If no history of illness is disclosed, tele-medical underwriting or a simple questionnaire may suffice for lower SAs.
*   **Mandatory Medical Tests (Standard for SA > ₹50 Lakhs or Age > 45):**
    *   **Physical:** Height, weight (BMI check), and Blood Pressure.
    *   **Pathology:** Fasting Blood Sugar (FBS), Lipid Profile, Liver Function Test (LFT), Kidney Function Test (KFT), and HbA1c.
    *   **Cardiac:** ECG (mandatory for all applicants > 45 years).
    *   **Urine:** Routine examination.
*   **Note:** Any "Abnormal" findings will trigger a manual referral to the Chief Medical Officer (CMO).

---

### 3. Financial Underwriting
The Sum Assured must be proportionate to the applicant's income to prevent over-insurance.

*   **Income Proof Requirements:**
    *   **Salaried:** Last 3 months' salary slips and latest Form 16/ITR.
    *   **Self-Employed:** Last 2 years’ ITR with computation of income and audited balance sheet.
*   **Financial Eligibility Ratio:**
    *   Age 25–35: Up to 20x annual income.
    *   Age 36–45: Up to 15x annual income.
    *   Age 46–55: Up to 10x annual income.
*   **Non-standard income:** Applicants with no verifiable income or income below the threshold will be declined.

---

### 4. Smoker Rules
*   **Declaration:** Applicant must declare tobacco/nicotine use (cigarettes, bidi, gutka, pan masala, or nicotine patches).
*   **Definition:** Any use of tobacco products within the last 12 months classifies the applicant as a "Smoker."
*   **Loading:** 
    *   Smokers will be subject to a higher premium rate (typically 20%–40% loading depending on the frequency of consumption).
    *   **Cotinine Test:** Mandatory for applicants where the smoker status is doubtful or for high Sum Assured cases.
*   **Misrepresentation:** If non-disclosure of smoking status is detected at the claim stage, the insurer reserves the right to repudiate the claim.

---

### 5. Pre-Existing Diseases (PED) Handling
*   **Definition:** Any medical condition for which the applicant has sought advice, diagnosis, or treatment in the 48 months preceding the policy application.
*   **Disclosure:** Applicants must mandatorily disclose all PEDs (e.g., Diabetes, Hypertension, Asthma, Thyroid issues, Cancer history).
*   **Treatment Categories:**
    *   **Low Risk (e.g., controlled Thyroid, minor allergies):** May be accepted at standard rates.
    *   **Moderate Risk (e.g., Type 2 Diabetes, controlled Hypertension):** May be accepted with "Extra Mortality Premium" (EMP) loading.
    *   **High Risk (e.g., Chronic Heart Disease, Cancer history):** Generally subject to declinature or heavy waiting periods.
*   **Exclusions:** Conditions specifically excluded by the policy wording will not be covered.
*   **Moratorium Period:** The insurer may impose a specific waiting period for certain pre-existing conditions before coverage for those conditions commences.

---

**Disclaimer:** *These guidelines are for informational purposes. Actual underwriting policies are subject to the individual insurer's actuarial data, regulatory requirements (IRDAI), and internal risk appetite.*


END OF MODEL RESPONSE


## 3. Demo 3 — ABC Health board summary (Health)

**Slide:** S1.P2.25
**Goal:** Take a JSON dict of model results from a hospitalisation-frequency
model and produce a 200-word executive summary suitable for the ABC Health
board pack.
**Output format:** Markdown, ~200 words, saved to `/content/demo_3_output.md`.

The pattern: structured numbers in, structured prose out. No statistic in the
output that wasn't in the input.


In [6]:
import json
from google import genai
from IPython.display import Markdown, display

# Create Gemini client
client = genai.Client()

# Select model
MODEL_ID = "gemini-3.1-flash-lite"

# Example health insurance model results
model_results = {
    "model_name": "Hospitalisation Risk Model",
    "members_modelled": 120000,
    "model_accuracy_auc": 0.81,
    "high_risk_member_lift": 3.2,
    "top_risk_drivers": [
        "Member Age",
        "Diabetes History",
        "Previous Hospitalisation"
    ],
    "fairness_gap_percent": 1.8
}

# Prompt
prompt = f"""
Write an executive summary for the board of a health insurance company.

Use the following model results:
{json.dumps(model_results, indent=2)}

Requirements:
- Around 150 words
- Use simple business language
- Explain the model performance clearly
- Mention the top risk drivers
- Mention the fairness gap
- End with a recommendation
- Return markdown only
"""

# Generate response
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
)

# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

# Render markdown nicely in Colab
display(Markdown(response.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE


### Executive Summary: Hospitalisation Risk Model

We have successfully developed and validated the **Hospitalisation Risk Model**, designed to proactively identify members at the highest risk of future hospital admissions. Applied to a cohort of 120,000 members, the model demonstrates strong predictive performance with an AUC of 0.81. Most notably, it achieves a **3.2x lift** in identifying high-risk individuals compared to our previous methods, allowing for more precise clinical intervention.

Our analysis confirms that the primary drivers of hospitalisation risk are **Member Age, Diabetes History, and Previous Hospitalisation**. 

In line with our commitment to equitable care, we conducted a fairness audit, which identified a minor **1.8% fairness gap** across demographics. This is well within acceptable regulatory and ethical thresholds. 

**Recommendation:** We recommend immediate integration of this model into our care management workflows. By prioritizing high-risk members identified by the engine, we can improve health outcomes, reduce avoidable emergency admissions, and optimize resource allocation.


END OF MODEL RESPONSE


## Wrap-up

You should now be able to:

- Call the Gemini API from Colab with a pinned model identifier and an API key loaded from Colab Secrets.
- Run a structured-output prompt that returns parseable JSON (Demo 1).
- Run a freeform-markdown prompt with explicit structure, length, and voice constraints (Demo 2).
- Feed a JSON of model results into a prompt and produce a board-ready prose summary that uses only the numbers you fed in (Demo 3).


**Where to next:** open `05_agents_intro.ipynb` for the minimal Agno + Gemini hello-agent used in Session 1 Part 4 and Session 2 Part 1.

**Companion slides:** S1.P2.23 (Demo 1), S1.P2.24 (Demo 2), S1.P2.25 (Demo 3) of the Session 1 deck (S1P2_Revised.pptx).

**Validation reminder:** The actuary owns the number, not the model. Before any output here goes near a regulatory deliverable, verify every figure against a real source.
